# 02 · Vector geometry — cross-persona transfer

**Core question:** is a trait encoded the *same way* across personas? We answer it
with the **transfer matrix** — pairwise cosine similarity of steering vectors.
High off-diagonal = universal; low = persona-specific.

In [ ]:
import torch
from pathlib import Path
from persona_steering.utils import parse_persona_trait_from_stem

def load_vector_dir(vectors_dir, layer=22):
    """Load all steering vectors from a dir of .pt files at one layer.

    Robust to two on-disk formats:
      * dict payload  {'vector': Tensor[L, D], 'persona':..., 'trait':...}
      * a raw Tensor[L, D]
    Returns {(persona, trait): Tensor[D]} for the requested layer.
    """
    vectors_dir = Path(vectors_dir)
    out = {}
    for pt in sorted(vectors_dir.glob('*.pt')):
        obj = torch.load(pt, map_location='cpu', weights_only=False)
        vec = obj['vector'] if isinstance(obj, dict) and 'vector' in obj else obj
        vec = vec.float()
        persona, trait = parse_persona_trait_from_stem(pt.stem)
        if persona and trait and layer < vec.shape[0]:
            out[(persona, trait)] = vec[layer]
    return out

In [ ]:
from pathlib import Path
from collections import defaultdict
from persona_steering.config import Trait
from persona_steering.utils import VectorShim
from persona_steering.analysis import build_transfer_matrix, build_per_trait_transfer
import numpy as np

DATA, LAYER = Path('hf_data'), 22
flat = load_vector_dir(DATA / 'vectors', layer=LAYER)  # (persona, trait) -> Tensor

# build_transfer_matrix wants nested: persona -> trait -> layer -> obj with .vector
nested = defaultdict(lambda: defaultdict(dict))
for (p, t), vec in flat.items():
    nested[p][Trait(t)][LAYER] = VectorShim(vec, p, Trait(t), LAYER)

personas = sorted(nested)
traits = [Trait(t) for t in sorted({t for (_, t) in flat})]
print(len(personas), 'personas x', len(traits), 'traits')

In [ ]:
M = build_transfer_matrix(nested, personas, traits, LAYER)
off_diag = M[~np.eye(len(personas), dtype=bool)]
print(f'mean off-diagonal cosine similarity: {off_diag.mean():.3f}')
print('(IV vectors are predominantly trait-universal but not identical across personas)')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(M, vmin=0, vmax=1, cmap='RdYlBu_r')
ax.set_xticks(range(len(personas))); ax.set_xticklabels(personas, rotation=90, fontsize=8)
ax.set_yticks(range(len(personas))); ax.set_yticklabels(personas, fontsize=8)
ax.set_title(f'Cross-persona transfer (mean over traits, layer {LAYER})')
fig.colorbar(im, label='cosine similarity'); plt.tight_layout(); plt.show()

### Per-trait view

Universality varies by trait. Honesty is typically the most universal; impulsivity
and risk-taking the most persona-specific.

In [ ]:
rows = []
for t in traits:
    pm = build_per_trait_transfer(nested, personas, t, LAYER)
    od = pm[~np.eye(len(personas), dtype=bool)]
    rows.append((t.value, od.mean()))
for name, val in sorted(rows, key=lambda r: -r[1]):
    print(f'  {name:14s} mean off-diag cos = {val:.3f}')

➡️ Next: **[03 · Shared vs specific decomposition](03_shared_specific.ipynb)** quantifies
exactly how much of each trait is universal.